# Test-Driven Development with AI Agents

**THE TDD WORKFLOW WITH AGENTS:**

1. **YOU** write the tests (you define what "correct" means)
2. You give the tests to the agent and say "implement code to pass these"
3. The agent writes the implementation
4. You run the tests
5. If tests fail, paste the errors back to the agent and let it fix

**WHY THIS IS POWERFUL:**
- Tests are unambiguous specifications. No room for misinterpretation.
- You don't need to know HOW to implement something, just WHAT it should do.
- You catch bugs immediately.
- The agent has a clear, measurable goal: make the tests pass.

**THIS NOTEBOOK DEMONSTRATES:**
- Part 1: The tests we'd write first (these define the specification)
- Part 2: The implementation an agent would generate to pass the tests
- Part 3: How to prompt the agent to do this

In [ ]:
import json

## Part 1: Write the Tests First

Before writing any implementation code, we define exactly what we want. These tests serve as the "specification" we'll hand to the agent.

In a real project, you'd use pytest. Here we use simple assert statements so the example is self-contained and easy to follow.

In [ ]:
def run_tests(PropertyAnalyzer):
    """
    Test suite for the PropertyAnalyzer class.

    These tests define WHAT the class should do, not HOW.
    The agent's job is to figure out the HOW.
    """
    passed = 0
    failed = 0

    # --- Test 1: Calculate price per square foot ---
    try:
        analyzer = PropertyAnalyzer()
        result = analyzer.price_per_sqft(500000, 1000)
        assert result == 500.0, f"Expected 500.0, got {result}"
        print("  PASS: price_per_sqft basic calculation")
        passed += 1
    except Exception as e:
        print(f"  FAIL: price_per_sqft basic calculation - {e}")
        failed += 1

    # --- Test 2: Price per sqft with zero sqft should raise ValueError ---
    try:
        analyzer = PropertyAnalyzer()
        try:
            analyzer.price_per_sqft(500000, 0)
            print("  FAIL: price_per_sqft zero sqft - should have raised ValueError")
            failed += 1
        except ValueError:
            print("  PASS: price_per_sqft zero sqft raises ValueError")
            passed += 1
    except Exception as e:
        print(f"  FAIL: price_per_sqft zero sqft - {e}")
        failed += 1

    # --- Test 3: Analyze comparables and return average price ---
    try:
        analyzer = PropertyAnalyzer()
        comps = [
            {"price": 500000, "sqft": 1000},
            {"price": 600000, "sqft": 1200},
            {"price": 700000, "sqft": 1400},
        ]
        result = analyzer.average_comp_price(comps)
        assert result == 600000.0, f"Expected 600000.0, got {result}"
        print("  PASS: average_comp_price calculation")
        passed += 1
    except Exception as e:
        print(f"  FAIL: average_comp_price calculation - {e}")
        failed += 1

    # --- Test 4: Average comp price with empty list should raise ValueError ---
    try:
        analyzer = PropertyAnalyzer()
        try:
            analyzer.average_comp_price([])
            print("  FAIL: average_comp_price empty list - should have raised ValueError")
            failed += 1
        except ValueError:
            print("  PASS: average_comp_price empty list raises ValueError")
            passed += 1
    except Exception as e:
        print(f"  FAIL: average_comp_price empty list - {e}")
        failed += 1

    # --- Test 5: Suggest listing price based on comps and property sqft ---
    try:
        analyzer = PropertyAnalyzer()
        comps = [
            {"price": 500000, "sqft": 1000},  # $500/sqft
            {"price": 600000, "sqft": 1000},  # $600/sqft
        ]
        # Average $/sqft = $550. Subject is 1200 sqft. Suggested = $660,000
        result = analyzer.suggest_listing_price(comps, subject_sqft=1200)
        assert result == 660000.0, f"Expected 660000.0, got {result}"
        print("  PASS: suggest_listing_price calculation")
        passed += 1
    except Exception as e:
        print(f"  FAIL: suggest_listing_price calculation - {e}")
        failed += 1

    # --- Test 6: Price range should return low and high estimates ---
    try:
        analyzer = PropertyAnalyzer()
        comps = [
            {"price": 500000, "sqft": 1000},
            {"price": 600000, "sqft": 1000},
        ]
        result = analyzer.price_range(comps, subject_sqft=1200, margin=0.05)
        # Suggested = $660,000. Range = $627,000 to $693,000
        assert result == (627000.0, 693000.0), f"Expected (627000.0, 693000.0), got {result}"
        print("  PASS: price_range with 5% margin")
        passed += 1
    except Exception as e:
        print(f"  FAIL: price_range with 5% margin - {e}")
        failed += 1

    # --- Test 7: Format price should return dollar string ---
    try:
        analyzer = PropertyAnalyzer()
        result = analyzer.format_price(1250000)
        assert result == "$1,250,000", f"Expected '$1,250,000', got '{result}'"
        print("  PASS: format_price")
        passed += 1
    except Exception as e:
        print(f"  FAIL: format_price - {e}")
        failed += 1

    # --- Summary ---
    print(f"\n  Results: {passed} passed, {failed} failed, {passed + failed} total")
    return failed == 0

## Part 2: The Implementation (what the agent would generate)

In a real TDD workflow, you would NOT write this yourself. You'd:
1. Write the tests above
2. Give them to the agent with: "Implement a PropertyAnalyzer class that passes all these tests"
3. The agent generates the code below

We include the implementation here so you can run the full example.

In [ ]:
class PropertyAnalyzer:
    """
    Property analysis utility for comparing listings against comparable sales.

    This implementation was generated to pass the test suite defined above.
    Each method corresponds to a specific test case.
    """

    def price_per_sqft(self, price: float, sqft: float) -> float:
        """Calculate price per square foot."""
        if sqft == 0:
            raise ValueError("Square footage cannot be zero")
        return price / sqft

    def average_comp_price(self, comps: list[dict]) -> float:
        """Calculate the average price from a list of comparable sales."""
        if not comps:
            raise ValueError("Comparables list cannot be empty")
        total = sum(comp["price"] for comp in comps)
        return total / len(comps)

    def suggest_listing_price(self, comps: list[dict], subject_sqft: float) -> float:
        """
        Suggest a listing price based on comparable sales and subject property sqft.
        Uses average price per sqft from comps, applied to subject sqft.
        """
        if not comps:
            raise ValueError("Comparables list cannot be empty")

        # Calculate average $/sqft across all comps
        avg_price_per_sqft = sum(
            comp["price"] / comp["sqft"] for comp in comps
        ) / len(comps)

        return avg_price_per_sqft * subject_sqft

    def price_range(
        self, comps: list[dict], subject_sqft: float, margin: float = 0.05
    ) -> tuple[float, float]:
        """
        Return a (low, high) price range based on suggested price and margin.
        Default margin is 5% above and below.
        """
        suggested = self.suggest_listing_price(comps, subject_sqft)
        low = suggested * (1 - margin)
        high = suggested * (1 + margin)
        return (low, high)

    def format_price(self, price: float) -> str:
        """Format a price as a dollar string with commas."""
        return f"${price:,.0f}"

## Part 3: The Prompt You'd Give to the Agent

This is the actual prompt you'd use in Claude Code or the Anthropic API to get the agent to generate the implementation.

In [ ]:
AGENT_PROMPT = """
Here is a test suite for a PropertyAnalyzer class. Implement the class so
that ALL tests pass. Do not modify the tests.

```python
{tests}
```

Requirements:
- Implement the PropertyAnalyzer class with all required methods
- Handle edge cases as defined by the tests (zero values, empty lists)
- Use type hints
- Add docstrings to each method
- Keep the implementation clean and simple

Run the tests to verify your implementation passes.
"""

print("The prompt we'd give to the agent:")
print(AGENT_PROMPT.format(tests="<the test code from Part 1 above>"))

## Run the Tests

Now we run the tests to verify the implementation is correct. This is the final step of the TDD workflow.

In [ ]:
print("=" * 60)
print("TDD EXAMPLE: PropertyAnalyzer")
print("=" * 60)
print()
print("WORKFLOW RECAP:")
print("  1. We wrote the tests FIRST (defining the specification)")
print("  2. We gave the tests to the agent")
print("  3. The agent generated the PropertyAnalyzer implementation")
print("  4. Now we run the tests to verify:")
print()

all_passed = run_tests(PropertyAnalyzer)

print()
if all_passed:
    print("ALL TESTS PASSED -- The agent's implementation is correct!")
else:
    print("SOME TESTS FAILED -- We'd paste these errors back to the agent")
    print("and ask it to fix the implementation.")